# Benchmarking

This benchmarking compares the PyTorch-native model variants (as opposed to the slowfast-based
MViTv2\_\*\_16x4/\_32x3 variants benchmarked in `sacair_2026/benchmark.ipynb`):
- S3D
- R3D_18
- R(2+1)D_18
- Swin3D_T
- Swin3D_S
- Swin3D_B
- MViTv2_S
- MViTv2_S_e
- MViTv1_B

each at 16 and/or 32 frames (models with interpolated positional encodings, suffix `_e`, use 32
frames; all others use 16, except the R3D/R(2+1)D/S3D/Swin3D families, which were benchmarked at
both), swept over batch size until each model OOMs -- the largest batch size shown per
model/frame-count/mode is therefore the last one that still fit.

During training, a stable GPU clock speed range of 1910 - 1930 MHz was recorded, so the GPU clock
speed is locked with a slight margin at 1900 with the command:
```bash
sudo nvidia-smi -lgc 1900
```
and can be reset with:
```bash
sudo nvidia-smi -rgc
```


In [1]:
import json

import pandas as pd

from src.run_types import RESULTS_DIR

In [2]:
def load_pytorch_benchmarks(benchmark_path):
    """
    Load the shared all_benchmark.json, keep only the PyTorch-native model variants (i.e. not
    the slowfast-based MViTv2_*_16x4/_32x3 variants covered by sacair_2026/benchmark.ipynb),
    drop OOM runs, map model names, and return separate DataFrames for training and inference.
    """
    with open(benchmark_path, "r") as f:
        raw = json.load(f)

    runs = raw["runs"]

    # Model name mapping
    model_name_map = {
        "S3D": "S3D",
        "R3D_18": "R3D\\_18",
        "R(2+1)D_18": "R(2+1)D\\_18",
        "Swin3D_T": "Swin3D\\_T",
        "Swin3D_S": "Swin3D\\_S",
        "Swin3D_B": "Swin3D\\_B",
        "MViTv2_S": "MViTv2\\_S",
        "MViTv2_S_e": "MViTv2\\_S\\_e",
        "MViTv1_B": "MViTv1\\_B",
    }

    records = []

    for run in runs.values():
        arch = run.get("arch", "")
        if arch not in model_name_map:
            continue
        if "error" in run:
            continue

        config = run["config"]
        results = run["results"]

        records.append({
            "model": model_name_map[arch],  # mapped name
            "num_frames": config["num_frames"],
            "mode": "Train" if config.get("full_step", False) else "Infer",
            "batch_size": config["batch_size"],

            "gpu_util_mean": results["gpu_utilisation_percent"]["mean"],
            "gpu_util_std": results["gpu_utilisation_percent"]["std"],

            "latency_ms_mean": results["latency_ms"]["mean"],
            "latency_ms_std": results["latency_ms"]["std"],

            "throughput_samp_per_s_mean": results["throughput_samples_per_s"]["mean"],
            "throughput_samp_per_s_std": results["throughput_samples_per_s"]["std"],

            "peak_mem_mb_mean": results["peak_memory_mb"]["mean"],
            "peak_mem_mb_std": results["peak_memory_mb"]["std"],
        })

    df = pd.DataFrame(records)

    metric_cols = [
        "gpu_util_mean", "gpu_util_std",
        "latency_ms_mean", "latency_ms_std",
        "throughput_samp_per_s_mean", "throughput_samp_per_s_std",
        "peak_mem_mb_mean", "peak_mem_mb_std",
    ]
    df[metric_cols] = df[metric_cols].round(2)

    # Sort by model, then num_frames, then batch_size
    train_df = df[df["mode"] == "Train"].sort_values(
        ["model", "num_frames", "batch_size"]
    ).reset_index(drop=True)
    infer_df = df[df["mode"] == "Infer"].sort_values(
        ["model", "num_frames", "batch_size"]
    ).reset_index(drop=True)

    return train_df, infer_df

## Load and display results

In [3]:
train_df, infer_df = load_pytorch_benchmarks(RESULTS_DIR / "all_benchmark.json")

print("Training DataFrame:")
display(train_df)

print("\nInference DataFrame:")
display(infer_df)

Training DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv1\_B,16,Train,1,99.99,0.02,127.93,0.11,7.82,0.01,1648.17,0.0
1,MViTv1\_B,16,Train,2,100.00,0.01,248.13,0.44,8.06,0.01,2952.42,0.0
2,MViTv1\_B,16,Train,4,100.00,0.00,471.92,0.16,8.48,0.00,5537.82,0.0
3,MViTv1\_B,16,Train,8,100.00,0.00,924.32,0.25,8.66,0.00,10707.35,0.0
4,MViTv2\_S,16,Train,1,98.66,0.02,175.50,0.39,5.70,0.01,1802.01,0.0
5,MViTv2\_S,16,Train,2,99.36,0.01,336.64,0.56,5.94,0.01,3294.64,0.0
6,MViTv2\_S,16,Train,4,99.79,0.02,643.22,0.72,6.22,0.01,6224.38,0.0
7,MViTv2\_S\_e,32,Train,1,99.49,0.01,427.50,0.65,2.34,0.00,3947.98,0.0
8,MViTv2\_S\_e,32,Train,2,99.87,0.01,837.06,0.92,2.39,0.00,7569.27,0.0
9,R(2+1)D\_18,16,Train,1,100.00,0.00,198.24,0.00,5.04,0.00,2274.04,0.0



Inference DataFrame:


,model,num_frames,mode,batch_size,gpu_util_mean,gpu_util_std,latency_ms_mean,latency_ms_std,throughput_samp_per_s_mean,throughput_samp_per_s_std,peak_mem_mb_mean,peak_mem_mb_std
0,MViTv1\_B,16,Infer,1,100.0,0.0,40.50,0.03,24.69,0.02,363.52,0.0
1,MViTv1\_B,16,Infer,2,100.0,0.0,78.52,0.10,25.47,0.03,565.19,0.0
2,MViTv1\_B,16,Infer,4,100.0,0.0,154.79,0.18,25.84,0.03,971.71,0.0
3,MViTv1\_B,16,Infer,8,100.0,0.0,304.31,0.33,26.29,0.03,1785.24,0.0
4,MViTv1\_B,16,Infer,16,100.0,0.0,604.40,0.80,26.47,0.04,3410.42,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
74,Swin3D\_T,32,Infer,1,100.0,0.0,81.98,0.10,12.20,0.01,831.13,0.0
75,Swin3D\_T,32,Infer,2,100.0,0.0,155.34,0.15,12.88,0.01,1445.30,0.0
76,Swin3D\_T,32,Infer,4,100.0,0.0,300.14,0.07,13.33,0.00,2679.43,0.0
77,Swin3D\_T,32,Infer,8,100.0,0.0,592.68,0.56,13.50,0.01,5145.68,0.0


### Prep for LaTeX

In [4]:
# Drop standard deviation columns and rename for table output
mean_cols = {
    "model": "Model",
    "num_frames": "Frames",
    "batch_size": "BS",
    "gpu_util_mean": "GPU Util.",
    "latency_ms_mean": "Latency (ms)",
    "throughput_samp_per_s_mean": "Throughput (samp/s)",
    "peak_mem_mb_mean": "Peak Mem. (MB)",
}

train_mean_df = train_df[list(mean_cols.keys())].rename(columns=mean_cols)
infer_mean_df = infer_df[list(mean_cols.keys())].rename(columns=mean_cols)

def format_benchmark_df(df):
    df = df.copy()
    df["GPU Util."] = df["GPU Util."].apply(lambda x: f"{x:.2f}\\%")
    df["Latency (ms)"] = df["Latency (ms)"].apply(lambda x: f"{x:,.2f}")
    df["Throughput (samp/s)"] = df["Throughput (samp/s)"].apply(lambda x: f"{x:,.2f}")
    df["Peak Mem. (MB)"] = df["Peak Mem. (MB)"].apply(lambda x: f"{x:,.2f}")
    return df

train_mean_df = format_benchmark_df(train_mean_df)
infer_mean_df = format_benchmark_df(infer_mean_df)

display(train_mean_df)
display(infer_mean_df)

,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv1\_B,16,1,99.99\%,127.93,7.82,"1,648.17"
1,MViTv1\_B,16,2,100.00\%,248.13,8.06,"2,952.42"
2,MViTv1\_B,16,4,100.00\%,471.92,8.48,"5,537.82"
3,MViTv1\_B,16,8,100.00\%,924.32,8.66,"10,707.35"
4,MViTv2\_S,16,1,98.66\%,175.50,5.70,"1,802.01"
5,MViTv2\_S,16,2,99.36\%,336.64,5.94,"3,294.64"
6,MViTv2\_S,16,4,99.79\%,643.22,6.22,"6,224.38"
7,MViTv2\_S\_e,32,1,99.49\%,427.50,2.34,"3,947.98"
8,MViTv2\_S\_e,32,2,99.87\%,837.06,2.39,"7,569.27"
9,R(2+1)D\_18,16,1,100.00\%,198.24,5.04,"2,274.04"


,Model,Frames,BS,GPU Util.,Latency (ms),Throughput (samp/s),Peak Mem. (MB)
0,MViTv1\_B,16,1,100.00\%,40.50,24.69,363.52
1,MViTv1\_B,16,2,100.00\%,78.52,25.47,565.19
2,MViTv1\_B,16,4,100.00\%,154.79,25.84,971.71
3,MViTv1\_B,16,8,100.00\%,304.31,26.29,"1,785.24"
4,MViTv1\_B,16,16,100.00\%,604.40,26.47,"3,410.42"
...,...,...,...,...,...,...,...
74,Swin3D\_T,32,1,100.00\%,81.98,12.20,831.13
75,Swin3D\_T,32,2,100.00\%,155.34,12.88,"1,445.30"
76,Swin3D\_T,32,4,100.00\%,300.14,13.33,"2,679.43"
77,Swin3D\_T,32,8,100.00\%,592.68,13.50,"5,145.68"


### Print LaTeX

In [5]:
def fixhlines(txt: str) -> str:
    return (
        txt.replace("\\toprule", "\\hline")
        .replace("\\midrule", "\\hline")
        .replace("\\bottomrule", "\\hline")
    )

In [6]:
train_latex = train_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Training benchmark results for PyTorch-native model variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:pytorch_benchmark_train",
    position="ht",
)

infer_latex = infer_mean_df.to_latex(
    index=False,
    escape=False,
    column_format="|l|c|c|r|r|r|r|",
    caption="Inference benchmark results for PyTorch-native model variants on an NVIDIA RTX 3060 12GB GPU.",
    label="tab:pytorch_benchmark_infer",
    position="ht",
)

print("Training Table LaTeX:")
print(fixhlines(train_latex))
print("\nInference Table LaTeX:")
print(fixhlines(infer_latex))

Training Table LaTeX:
\begin{table}[ht]
\caption{Training benchmark results for PyTorch-native model variants on an NVIDIA RTX 3060 12GB GPU.}
\label{tab:pytorch_benchmark_train}
\begin{tabular}{|l|c|c|r|r|r|r|}
\hline
Model & Frames & BS & GPU Util. & Latency (ms) & Throughput (samp/s) & Peak Mem. (MB) \\
\hline
MViTv1\_B & 16 & 1 & 99.99\% & 127.93 & 7.82 & 1,648.17 \\
MViTv1\_B & 16 & 2 & 100.00\% & 248.13 & 8.06 & 2,952.42 \\
MViTv1\_B & 16 & 4 & 100.00\% & 471.92 & 8.48 & 5,537.82 \\
MViTv1\_B & 16 & 8 & 100.00\% & 924.32 & 8.66 & 10,707.35 \\
MViTv2\_S & 16 & 1 & 98.66\% & 175.50 & 5.70 & 1,802.01 \\
MViTv2\_S & 16 & 2 & 99.36\% & 336.64 & 5.94 & 3,294.64 \\
MViTv2\_S & 16 & 4 & 99.79\% & 643.22 & 6.22 & 6,224.38 \\
MViTv2\_S\_e & 32 & 1 & 99.49\% & 427.50 & 2.34 & 3,947.98 \\
MViTv2\_S\_e & 32 & 2 & 99.87\% & 837.06 & 2.39 & 7,569.27 \\
R(2+1)D\_18 & 16 & 1 & 100.00\% & 198.24 & 5.04 & 2,274.04 \\
R(2+1)D\_18 & 16 & 2 & 100.00\% & 384.91 & 5.20 & 4,174.97 \\
R(2+1)D\_18 & 16 & 4